# Classification: Distributional Features

Instead of averaging each metric into a single mean value, we capture the distribution of values across trials using quartiles (Q25, Q50, Q75) and interquartile range (IQR). This preserves information about variability that session-level means lose — for example, whether a person's fixation bias was consistently moderate or swung between extremes.

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

from src.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance,
)

## 1. Build distributional features

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

BASE_METRICS = [
    "fixation_count",
    "mean_fixation_duration_ms",
    "total_fixation_duration_ms",
    "fixation_rate_per_sec",
    "fixation_bias",
    "scanpath_length",
    "saccade_count",
    "saccade_rate_per_sec",
    "mean_saccade_amplitude",
    "blink_count",
    "blink_rate_per_min",
    "transition_matrix_density",
    "gaze_transition_entropy",
    "first_fixation_duration_ms",
    "second_fixation_duration_ms",
    "dwell_time_ms_negative",
    "dwell_time_ms_positive",
    "dwell_time_ms_neutral",
    "dwell_time_500ms_negative",
    "dwell_time_500ms_positive",
    "dwell_time_500ms_neutral",
    "fixation_proportion_negative",
    "fixation_proportion_positive",
    "fixation_proportion_neutral",
    "fixation_count_negative",
    "fixation_count_positive",
    "fixation_count_neutral",
    "revisit_count_negative",
    "revisit_count_positive",
    "revisit_count_neutral",
    "ttff_ms_negative",
    "ttff_ms_positive",
    "ttff_ms_neutral",
]

agg_exprs = []
for m in BASE_METRICS:
    agg_exprs.append(F.percentile_approx(m, 0.25).alias(f"{m}_q25"))
    agg_exprs.append(F.percentile_approx(m, 0.50).alias(f"{m}_q50"))
    agg_exprs.append(F.percentile_approx(m, 0.75).alias(f"{m}_q75"))

for valence in ["negative", "positive", "neutral"]:
    agg_exprs.append(
        F.avg(F.when(F.col("first_fixation_valence") == valence, 1).otherwise(0))
         .alias(f"first_fix_prob_{valence}"))
    agg_exprs.append(
        F.avg(F.when(F.col("second_fixation_valence") == valence, 1).otherwise(0))
         .alias(f"second_fix_prob_{valence}"))

session_metrics = df_stimulus.groupBy("session_id").agg(*agg_exprs)

df_joined = session_metrics.join(
    df_forms.select("session_id", "uid", "phq9_score", "phq9_severity"),
    on="session_id", how="inner",
)

df = df_joined.toPandas()

# compute IQR in pandas
for m in BASE_METRICS:
    df[f"{m}_iqr"] = df[f"{m}_q75"] - df[f"{m}_q25"]

print(f"Sessions: {len(df)}, Users: {df['uid'].nunique()}")

## 2. Define targets and feature sets

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)
df["phq9_severity_class"] = df["phq9_severity"].map(
    {"minimal": 0, "mild": 1, "moderate": 2, "moderately_severe": 3, "severe": 4}
)

print(f"Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")
print(f"Multi-class: {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

In [0]:
QUARTILE_FEATURES = []
for m in BASE_METRICS:
    QUARTILE_FEATURES.extend([f"{m}_q25", f"{m}_q50", f"{m}_q75", f"{m}_iqr"])

PROB_FEATURES = [
    "first_fix_prob_negative", "first_fix_prob_positive", "first_fix_prob_neutral",
    "second_fix_prob_negative", "second_fix_prob_positive", "second_fix_prob_neutral",
]

ALL_FEATURES = QUARTILE_FEATURES + PROB_FEATURES

THEORY_BASE = [
    "fixation_bias", "dwell_time_ms_negative", "dwell_time_ms_positive",
    "fixation_proportion_negative", "fixation_proportion_positive",
    "revisit_count_negative", "revisit_count_positive",
    "blink_rate_per_min", "scanpath_length",
]
THEORY_FEATURES = []
for m in THEORY_BASE:
    THEORY_FEATURES.extend([f"{m}_q25", f"{m}_q50", f"{m}_q75", f"{m}_iqr"])
THEORY_FEATURES.extend(["first_fix_prob_negative", "first_fix_prob_positive"])

MEDIAN_FEATURES = [f"{m}_q50" for m in BASE_METRICS] + PROB_FEATURES

FEATURE_SETS = {
    "All Distributional": ALL_FEATURES,
    "Theory-Driven Distributional": THEORY_FEATURES,
    "Median Only": MEDIAN_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = ["phq9_score", "phq9_depressed", "phq9_severity_class"]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. Binary classification (PHQ-9 >= 10)

### 4.1 Run classification

In [0]:
y_binary = df_clean["phq9_depressed"].values
binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_binary, groups)

### 4.2 Results

In [0]:
print(binary_df.to_string(index=False))

### 4.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_binary, groups, binary_df)

## 5. Multi-class classification (5 severity groups)

### 5.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_multi = df_clean["phq9_severity_class"].values.astype(int)
multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_multi, groups)

### 5.2 Results

In [0]:
print(multi_df.to_string(index=False))

### 5.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_multi, groups, multi_df, PHQ9_LABELS)

## 6. Regression (predict PHQ-9 score)

### 6.1 Run regression

In [0]:
y_reg = df_clean["phq9_score"].values
reg_df = run_regression(df_clean, FEATURE_SETS, y_reg, groups)

### 6.2 Results

In [0]:
print(reg_df.to_string(index=False))

### 6.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_reg, groups, reg_df)

## 7. Feature importance

In [0]:
plot_feature_importance(df_clean, ALL_FEATURES, y_binary, title="Feature importance (binary, all distributional)")

## 8. Summary

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(binary_df, multi_df, reg_df, feature_order, title="Distributional features")

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

feat_order = ["All Distributional", "Theory-Driven Distributional", "Median Only"]

pivot_binary = binary_df.pivot(index="model", columns="feature_set", values="auc_roc")
pivot_binary = pivot_binary[feat_order]
pivot_binary.plot(kind="bar", ax=axes[0], rot=15, edgecolor="white")
axes[0].set_title("Binary — AUC-ROC")
axes[0].set_ylabel("AUC-ROC")
axes[0].set_ylim(0.45, 0.75)
axes[0].axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Random")
axes[0].legend(fontsize=7)

pivot_multi = multi_df.pivot(index="model", columns="feature_set", values="f1_weighted")
pivot_multi = pivot_multi[feat_order]
pivot_multi.plot(kind="bar", ax=axes[1], rot=15, edgecolor="white")
axes[1].set_title("Multi-class — Weighted F1")
axes[1].set_ylabel("Weighted F1")

pivot_reg = reg_df.pivot(index="model", columns="feature_set", values="r2")
pivot_reg = pivot_reg[feat_order]
pivot_reg.plot(kind="bar", ax=axes[2], rot=15, edgecolor="white")
axes[2].set_title("Regression — R2")
axes[2].set_ylabel("R2")
axes[2].axhline(y=0, color="red", linestyle="--", alpha=0.5)

plt.suptitle("Distributional Features — Model Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()